# MoroccoLaw-LLM — Setup and RAG Prototype

This notebook builds a small retrieval-augmented generation (RAG) prototype for Moroccan legal documents. It covers environment setup, PDF extraction, article parsing, multilingual embeddings, FAISS retrieval, grounded Qwen responses, and optional instruction-dataset generation.

> **Educational use only:** generated responses are not legal advice. Always verify answers against the cited official legal text.


## 1. Install dependencies

The notebook is designed for Google Colab but can also run in a local Jupyter environment. After the first installation, restart the runtime if Colab requests it.


In [ ]:
%pip install -q transformers datasets peft accelerate trl bitsandbytes sentence-transformers faiss-cpu pymupdf json-repair


In [ ]:
import json
import re
from pathlib import Path

import faiss
import fitz
import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. Configure storage

On Colab, the project is stored in Google Drive so generated files survive runtime resets. Locally, files are created under the repository root.


In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/MoroccoLaw-LLM")
except ImportError:
    current = Path.cwd()
    PROJECT_ROOT = current.parent if current.name == "notebooks" else current

RAW_DIR = PROJECT_ROOT / "datasets/raw/legal_documents"
CLEAN_DIR = PROJECT_ROOT / "datasets/cleaned"
PROCESSED_DIR = PROJECT_ROOT / "datasets/processed"
TRAINING_DIR = PROJECT_ROOT / "datasets/training"
MODELS_DIR = PROJECT_ROOT / "models"

for directory in (RAW_DIR, CLEAN_DIR, PROCESSED_DIR, TRAINING_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


## 3. Load the Qwen instruction model

The 0.5B model keeps this tutorial accessible in a free Colab session. Replace `MODEL_NAME` with a larger compatible Qwen model when more GPU memory is available.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

print(f"Loaded {MODEL_NAME} on {model.device}")


In [ ]:
def ask_qwen(prompt: str, max_new_tokens: int = 300) -> str:
    messages = [{"role": "user", "content": prompt}]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(rendered, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


print(ask_qwen("Explain what a contract is in one short paragraph."))


## 4. Extract and clean a Moroccan legal PDF

Place an authorized copy of `Code_obligations_contrats.pdf` in `datasets/raw/legal_documents/`. Legal source files are intentionally excluded from Git because their redistribution rights may vary.


In [ ]:
PDF_PATH = RAW_DIR / "Code_obligations_contrats.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PDF_PATH}. Add the legal PDF there, then rerun this cell."
    )

with fitz.open(PDF_PATH) as document:
    pdf_text = "\n".join(page.get_text("text") for page in document)

print(f"Extracted {len(pdf_text):,} characters")
print(pdf_text[:1_500])


In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


cleaned_text = clean_text(pdf_text)
article_one = re.search(r"\bArticle\s+1\b", cleaned_text, flags=re.IGNORECASE)
if article_one:
    cleaned_text = cleaned_text[article_one.start():]

CLEAN_TEXT_PATH = CLEAN_DIR / "DOC_clean.txt"
CLEAN_TEXT_PATH.write_text(cleaned_text, encoding="utf-8")

print(f"Saved cleaned text to {CLEAN_TEXT_PATH}")
print(cleaned_text[:1_500])


## 5. Parse the document into legal articles

Each article becomes one structured JSON record containing the law name, article number, language, and source text.


In [ ]:
ARTICLE_PATTERN = re.compile(
    r"(Article\s+\d+(?:-\d+)?[\s\S]*?)(?=Article\s+\d+(?:-\d+)?|$)",
    flags=re.IGNORECASE,
)

articles = ARTICLE_PATTERN.findall(cleaned_text)
legal_docs = []

for article_text in articles:
    number_match = re.search(
        r"Article\s+(\d+(?:-\d+)?)",
        article_text,
        flags=re.IGNORECASE,
    )
    if not number_match:
        continue

    legal_docs.append(
        {
            "law": "Code des Obligations et des Contrats",
            "article": number_match.group(1),
            "language": "fr",
            "text": article_text.strip(),
        }
    )

if not legal_docs:
    raise ValueError("No legal articles were detected. Inspect the extracted PDF text.")

LEGAL_DOCS_PATH = PROCESSED_DIR / "legal_documents.json"
LEGAL_DOCS_PATH.write_text(
    json.dumps(legal_docs, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(legal_docs)} articles to {LEGAL_DOCS_PATH}")
print(json.dumps(legal_docs[0], ensure_ascii=False, indent=2)[:1_500])


## 6. Create multilingual embeddings and a FAISS index

Embeddings are normalized and stored in an inner-product index, making the returned score equivalent to cosine similarity.


In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [document["text"] for document in legal_docs]
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

INDEX_PATH = MODELS_DIR / "legal_index.faiss"
faiss.write_index(index, str(INDEX_PATH))

print("Embedding shape:", embeddings.shape)
print("Indexed articles:", index.ntotal)
print("Saved index to:", INDEX_PATH)


## 7. Retrieve relevant articles


In [ ]:
def search_law(question: str, k: int = 3) -> list[dict]:
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = index.search(query_embedding, min(k, index.ntotal))
    results = []

    for score, idx in zip(scores[0], indices[0]):
        document = dict(legal_docs[int(idx)])
        document["score"] = float(score)
        results.append(document)

    return results


sample_results = search_law("Quelles sont les conditions de validité d'un contrat ?")
for result in sample_results:
    print(f"Article {result['article']} — score {result['score']:.3f}")


## 8. Generate a source-grounded answer

The prompt requires the model to stay within the retrieved articles, cite article numbers, and decline when the available context is insufficient.


In [ ]:
def build_grounded_prompt(question: str, documents: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source: {doc['law']} — Article {doc['article']}]\n{doc['text']}"
        for doc in documents
    )

    return f'''You are MoroccoLaw AI, an educational assistant specialized in Moroccan law.

Rules:
1. Answer only from the legal sources provided below.
2. Cite every legal claim using the relevant article number.
3. Explain the answer clearly and concisely.
4. If the sources are insufficient, say that you cannot answer from the available documents.
5. Do not present the response as professional legal advice.

Legal sources:
{context}

Question: {question}

Answer:'''


def answer_legal_question(question: str, k: int = 3) -> dict:
    sources = search_law(question, k=k)
    prompt = build_grounded_prompt(question, sources)
    answer = ask_qwen(prompt, max_new_tokens=400)
    return {"question": question, "answer": answer, "sources": sources}


In [ ]:
result = answer_legal_question(
    "Quelles sont les conditions de validité d'un contrat ?",
    k=4,
)

print(result["answer"])
print("\nSources retrieved:")
for source in result["sources"]:
    print(f"- {source['law']}, article {source['article']}")


## 9. Optional: generate an instruction dataset

This section creates candidate French question-answer examples from the legal articles. Generated examples must be reviewed against the original text before any fine-tuning experiment.


In [ ]:
from json_repair import repair_json


def build_training_prompt(document: dict) -> str:
    return f'''Create three French instruction-tuning examples from the legal article below.

Requirements:
- Use only information explicitly present in the article.
- Do not invent legal rules.
- Return only a valid JSON array.
- Each object must contain: instruction, input, and output.
- Keep input as an empty string.
- Mention the source article in each answer.

Law: {document['law']}
Article: {document['article']}
Text: {document['text']}
'''


def parse_generated_examples(response: str) -> list[dict]:
    parsed = json.loads(repair_json(response))
    if not isinstance(parsed, list):
        raise ValueError("The model response is not a JSON list.")

    required = {"instruction", "input", "output"}
    return [item for item in parsed if isinstance(item, dict) and required <= item.keys()]


In [ ]:
# Start with a small subset; increase only after manually checking quality.
MAX_ARTICLES = 5
training_examples = []

for document in legal_docs[:MAX_ARTICLES]:
    response = ask_qwen(build_training_prompt(document), max_new_tokens=700)
    try:
        examples = parse_generated_examples(response)
        for example in examples:
            example["source_law"] = document["law"]
            example["source_article"] = document["article"]
        training_examples.extend(examples)
    except (json.JSONDecodeError, ValueError) as error:
        print(f"Skipped article {document['article']}: {error}")

TRAINING_PATH = TRAINING_DIR / "legal_instruction_dataset.json"
TRAINING_PATH.write_text(
    json.dumps(training_examples, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(training_examples)} candidate examples to {TRAINING_PATH}")


In [ ]:
training_dataset = load_dataset(
    "json",
    data_files=str(TRAINING_PATH),
    split="train",
)

print(training_dataset)
print(training_dataset[0] if len(training_dataset) else "Dataset is empty")


## Next steps

- Add more official Moroccan legal sources with document-level metadata.
- Preserve page numbers during PDF extraction for verifiable citations.
- Evaluate retrieval with a reviewed set of legal questions.
- Add hybrid BM25 + dense retrieval.
- Validate generated answers and citations deterministically.
- Fine-tune only with legally reviewed, provenance-linked examples.
